# Run multigauge on your own basin

Enter a USGS streamgage number. This notebook downloads the basin boundary, streamflow, Daymet weather and the
NOAA rain gauges inside the basin, fine-tunes the pre-trained model, measures what the rain gauges add, and
gives you an HTML report. It takes about 15–30 minutes on a free GPU.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

To use your own rain gauges (for example a state mesonet) instead of NOAA's, upload `rainfall.csv` and
`stations.csv` in step 3.

In [ ]:
SITE = "07326500"          # USGS streamgage number
NAME = ""                  # optional basin name
START, END = "2000-10-01", "2023-09-30"
SEEDS = 3                  # models per configuration (3 recommended)

## 1. Install

In [ ]:
!rm -rf multigauge-hydro && git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git
!pip install -q -e "multigauge-hydro[download]"
import multigauge as mg, torch
print("GPU:", torch.cuda.is_available(), "| pre-trained models:", mg.available_pretrained())

## 2. Download the basin's data

In [ ]:
from pathlib import Path
from multigauge.prepare import prepare_usgs_basin
folder = Path(f"/content/basins/{SITE}")
data = prepare_usgs_basin(SITE, START, END, folder, name=NAME or None)
print(data.name, f"{data.area_km2:,.0f} km2,", len(data.station_ids), "gauges,", len(data.dates), "days")

## 3. (Optional) use your own rain gauges

Upload `rainfall.csv` (date + one column per gauge, mm/day) and `stations.csv` (id, lat, lon). Press Cancel to keep the NOAA gauges.

In [ ]:
from google.colab import files
up = files.upload()
if {"rainfall.csv", "stations.csv"} <= set(up):
    for f in ("rainfall.csv", "stations.csv"):
        (folder / f).write_bytes(up[f])
    data = mg.BasinData.from_folder(folder)
    print("Using your", len(data.station_ids), "gauges")

## 4. Run the gauge-value study

In [ ]:
from multigauge.report import summary_table
base = mg.load_pretrained("camels-us")
res = mg.run_study(data, base, mg.StudyConfig(n_seeds=SEEDS, finetune_epochs=20, run_ablation=True),
                   progress=lambda f, m: print(f"\r{int(f*100):3d}% {m[:80]:<80}", end=""))
print()
summary_table(res).round(3)

In [ ]:
res.gauge_value[res.gauge_value["period"] == "test"].round(3)

## 5. Report and download

In [ ]:
out = res.save(folder / "results")
mg.write_report(res, data, out / "report.html")
!cd /content/basins && zip -qr /content/multigauge_{SITE}.zip {SITE}
files.download(f"/content/multigauge_{SITE}.zip")